# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import torch

from experiments.models import HyperTreeNetARForecast

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
)

## Parameters

In [ ]:
# Empty cell for final run
data_run = "ABC"
embedding_dim = 0

# Load Experiment Specifications

In [ ]:
# Load specifications
dataset = data_run
train_type = "local" if dataset == "airpassengers" else "global"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

save_extras = dataset.lower() == "airpassengers" and embedding_dim == 10

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]

lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
series_ids = meta["series_ids"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
loss_fn = config["general"]["loss_function"]

dl_params = config["deep_learning"]
htnetar_params = config["hyper_treenet_params"]
htnetar_params_lgb = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "embedding_dimension", "use_random_projection"]}
htnetar_network_params = {k: v for k, v in htnetar_params.items() if k not in ["num_boost_round", "eta", "linear_tree"]}
htnetar_network_params["rp_embed_dim"] = max_lag
htnetar_network_params["hidden_dim"] = dl_params["hidden_size"]
htnetar_network_params["dropout"] = dl_params["dropout"]
htnetar_network_params["learning_rate"] = dl_params["learning_rate"]

# TS-Features

In [ ]:
# Extract time series features for global datasets only (not applicable for AirPassengers as it is a single series)
if dataset != "airpassengers":
    ts_feats_df, ts_feats = create_tsfeatures(
        train=train_df,
        freq=freq
    )

    # Add features to train and test
    train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
    test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
    features = features + ts_feats

# Hyper-TreeNet-AR

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

htnetar_network_params["embedding_dimension"] = embedding_dim

if save_extras:
    htnet_ar_fcst, htnet_ar_params, htnet_ar_embeddings = HyperTreeNetARForecast(
        htnetar_params,
        htnetar_params_lgb,
        htnetar_network_params,
        train_df,
        test_df.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        max_lag,
        loss_fn,
        seed,
        device,
        return_extras=True,
    )
else:
    htnet_ar_fcst = HyperTreeNetARForecast(
        htnetar_params,
        htnetar_params_lgb,
        htnetar_network_params,
        train_df,
        test_df.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        max_lag,
        loss_fn,
        seed,
        device,
    )
    htnet_ar_params = None
    htnet_ar_embeddings = None

# Add actual values and dataset information

In [ ]:
# Add actual values and dataset information
fcsts_df = pd.merge(
    htnet_ar_fcst,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)
fcsts_df["embedding_dim"] = embedding_dim

# Output

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / "ablation" / "embedding_evaluation"
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_{embedding_dim}_fcsts.csv", index=False)

# Tree embeddings + AR parameters (only for the visualization dataset).
if save_extras:
    htnet_ar_params.to_csv(
        result_path / f"{dataset.lower()}_{embedding_dim}_ar_parameters.csv",
        index=False,
    )
    htnet_ar_embeddings.to_csv(
        result_path / f"{dataset.lower()}_{embedding_dim}_tree_embeddings.csv",
        index=False,
    )